In [19]:
import pandas as pd
TRAIN_PATH = "../data/features/essay_train_smote.csv"
TEST_PATH = "../data/features/essay_test_scaled.csv"

train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)
test_data

,adapted_dalechall,adverbs_before_main_verb_ratio,brunet_indice,clauses_per_sentence,cncadc,cncadd,cncall,cncalter,cnccaus,cnccomp,...,wrdprp1p,wrdprp1s,wrdprp2,wrdprp2p,wrdprp2s,wrdprp3p,wrdprp3s,wrdverb,yule_k,target
0,-1.347021,-0.472203,0.515769,-0.601238,0.484122,2.789352,2.585322,-0.951121,0.399652,-0.029367,...,-0.181715,-0.084029,-0.128131,-0.034467,-0.124436,-0.456054,-0.469185,0.682136,-0.097384,2
1,2.198038,-0.094002,-0.845537,2.297294,-1.104210,-0.358636,-0.660713,-0.951121,-0.723540,-1.163243,...,-0.181715,-0.084029,-0.128131,-0.034467,-0.124436,-0.456054,-0.469185,-1.136700,-0.453588,2
2,-0.748844,-0.158640,0.694646,-0.553634,0.484122,1.215358,1.165182,-0.276757,0.175013,0.650958,...,-0.181715,-0.084029,-0.128131,-0.034467,-0.124436,-0.456054,-0.469185,1.028580,0.090033,3
3,-0.273935,-0.708062,-0.519672,-0.262723,0.484122,-0.183748,-0.052081,-0.951121,-0.498902,-0.709693,...,-0.181715,-0.084029,-0.128131,-0.034467,-0.124436,-0.456054,-0.469185,-0.703644,-0.278633,3
4,0.741819,1.049592,-1.005242,-0.169632,-1.104210,-0.008859,-0.457836,0.397606,-0.948178,-0.936468,...,-0.181715,-0.084029,-0.128131,-0.034467,-0.124436,-0.456054,-0.469185,-0.876866,-0.294454,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1308,-0.289360,1.037161,-0.970195,-1.141921,-0.310044,0.166029,-0.254958,-0.951121,-0.723540,-0.936468,...,-0.181715,-0.084029,-0.128131,-0.034467,-0.124436,-0.456054,-0.469185,-0.703644,-0.538463,3
1309,0.093935,1.931319,-0.538516,-0.844545,-1.104210,-0.708412,-0.863590,-0.951121,0.399652,0.650958,...,-0.181715,-0.084029,-0.128131,-0.034467,-0.124436,-0.456054,-0.469185,-0.270588,-0.473280,3
1310,-0.486612,-0.481830,0.578151,0.371992,-0.310044,-0.708412,-0.863590,-0.276757,0.848928,0.650958,...,-0.181715,-0.084029,-0.128131,-0.034467,-0.124436,-0.456054,-0.469185,0.335691,-0.126376,2
1311,-0.499639,-0.946864,0.195966,-0.355815,-1.104210,0.340917,-0.254958,0.397606,0.624290,0.650958,...,-0.181715,-0.084029,-0.128131,-0.034467,-0.124436,-0.456054,-0.469185,-0.443810,0.305634,4


In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

feature_names = train_data.columns[:-1]
class_names = [0, 0.5, 1., 1.5, 2.]

X_train = train_data.loc[:, feature_names].to_numpy()
Y_train = train_data.target.to_numpy()

X_test = test_data.loc[:, feature_names].to_numpy()
Y_test = test_data.target.to_numpy()

In [21]:
rf_clf = RandomForestClassifier(max_depth=12, random_state=0).fit(X_train, Y_train)
#svc_clf = SVC(kernel="linear",probability=True).fit(X_train, Y_train)
lr_clf = LogisticRegression(random_state=0).fit(X_train, Y_train)
clf_list = [rf_clf,lr_clf ]

/opt/conda/envs/aibox-env/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [4]:

# Calculando a incerteza pela máxima probabilidade
def calculate_uncertanty(clf, x_test):
      probs = clf.predict_proba(x_test)
      max_probs = probs.max(axis=1)
      uncertainty = 1 - max_probs
      return uncertainty

In [5]:
import pandas as pd
import numpy as np

uncertanties = {}
for model in  clf_list:

  uncertanties[model.__class__.__name__] = calculate_uncertanty(model, X_test)


uncertanty_df = pd.DataFrame(uncertanties).set_index(np.array([f"x{i}" for i in range(X_test.shape[0])]))
uncertanty_df

,RandomForestClassifier,LogisticRegression
x0,0.337021,0.313931
x1,0.332969,0.295351
x2,0.539143,0.517882
x3,0.454359,0.519876
x4,0.305732,0.488030
...,...,...
x1308,0.577397,0.299100
x1309,0.503160,0.574474
x1310,0.518454,0.456528
x1311,0.422120,0.234003


In [6]:
uncertanty_df.agg(['mean', 'var'], axis=1).rename(columns={
    'mean': "Incerteza Aleatórica",
    'var': "Incerteza Epstêmica"
})

,Incerteza Aleatórica,Incerteza Epstêmica
x0,0.325476,0.000267
x1,0.314160,0.000708
x2,0.528513,0.000226
x3,0.487118,0.002146
x4,0.396881,0.016616
...,...,...
x1308,0.438249,0.038725
x1309,0.538817,0.002543
x1310,0.487491,0.001917
x1311,0.328061,0.017694


In [32]:
start_feat = 3
end_feat = 5
features = test_data.columns[:-1]
selected_features = ["adverbs_before_main_verb_ratio",	"clauses_per_sentence"]

X_test_selected = X_train[:, np.isin(features, selected_features)]
X_train_selected = X_train[:, np.isin(features, selected_features)]
X_test_selected = X_train[:, np.isin(features, selected_features)]

rf_selected = RandomForestClassifier().fit(X_train_selected, Y_train)
lr_selected = LogisticRegression(random_state=0).fit(X_train_selected, Y_train)

clf_list_selected = [ rf_selected, lr_selected]

In [33]:
uncertanties = {  }
for model in  clf_list_selected:
  uncertanties[model.__class__.__name__] = calculate_uncertanty(model, X_test_selected)

uncertanty_df = pd.DataFrame(uncertanties).set_index(np.array([f"x{i}" for i in range(X_test_selected.shape[0])]))

In [36]:
agg_uncertanty_df = uncertanty_df.agg(['mean', 'var'], axis=1).rename(columns={
    'mean': "Incerteza Aleatórica",
    'var': "Incerteza Epstêmica"
})
agg_uncertanty_df

,Incerteza Aleatórica,Incerteza Epstêmica
x0,0.681389,0.023786
x1,0.538572,0.027646
x2,0.526485,0.128172
x3,0.382788,0.171450
x4,0.583013,0.031591
...,...,...
x10525,0.577757,0.056285
x10526,0.567427,0.056063
x10527,0.510972,0.097657
x10528,0.512226,0.159304


In [38]:
feats = {
    
}

for i in selected_features:
    print(i)
    feats[i] = X_test[:, np.isin(features, [i])]

agg_uncertanty_df = pd.concat([agg_uncertanty_df, pd.DataFrame(feats).set_index(np.array([f"x{i}" for i in range(X_test.shape[0])])) ], axis=1)
agg_uncertanty_df

[-0.60123775  0.4841221 ]


TypeError: unhashable type: 'numpy.ndarray'

In [35]:
import seaborn as sns
sns.scatterplot(data=agg_uncertanty_df, x=selected_features[0], y=selected_features[1], hue="Incerteza Aleatórica")

ValueError: Could not interpret value `adverbs_before_main_verb_ratio` for `x`. An entry with this name does not appear in `data`.